# 09 · Convolution and deconvolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/09-convolution-and-deconvolution.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Convolución y deconvolución** — Ver la convolución como un operador lineal estructurado, distinguir correlación de convolución, entender la convolución transpuesta y recuperar parcialmente una imagen real desenfocada.

Treat convolution as a structured linear operator, separate it from correlation, understand transposed convolution, and partially recover a real blurred image.

## What you will be able to do

- Predict `full`, `same`, and `valid` output shapes for a real image and verify them interactively.
- Explain why deep-learning convolution is usually cross-correlation and show the kernel-flip relationship.
- Write a 1D convolution of a real image scanline as multiplication by a Toeplitz matrix.
- Explain transposed convolution as an overlap-add linear operator that changes shape but is not a true inverse.
- Recover a real blurred image with Richardson-Lucy and measure improvement away from boundary artifacts.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from scipy import signal
from scipy.linalg import toeplitz
from skimage import data
from skimage.restoration import richardson_lucy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Enable ipywidgets in Google Colab when available.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

img = data.camera().astype(float) / 255.0
patch = img[176:336, 176:336]
work = img[128:384, 128:384]
scanline = img[256, 220:252].copy()

sobel_x = np.array([
    [-1., 0., 1.],
    [-2., 0., 2.],
    [-1., 0., 1.],
])

kernel_1d = np.array([1., 0., -1.])

def convmtx_full_1d(kernel, n):
    m = len(kernel)
    col = np.zeros(n + m - 1)
    col[:m] = kernel
    row = np.zeros(n)
    row[0] = kernel[0]
    return toeplitz(col, row)

print("full image / imagen completa:", img.shape)
print("real patch / recorte real:", patch.shape)
print("deconvolution crop / recorte deconvolución:", work.shape)
print("real scanline / fila real:", scanline.shape)

## Why this matters

Convolution is not a mysterious new kind of multiplication. It is a **linear operator with repeated structure**: the same small kernel is reused across positions.

That gives us three important connections:

- **convolution ↔ correlation:** true convolution flips the kernel; cross-correlation does not;
- **convolution ↔ matrix multiplication:** a Toeplitz matrix can represent the same operation;
- **blur ↔ inverse problem:** once convolution destroys or suppresses information, “undoing” it can be unstable.

A transposed convolution belongs to the second connection: it is the transpose/adjoint-style partner of a convolution operator. It can enlarge an array through overlap-add, but it does **not** reconstruct the original values by itself.

True deconvolution belongs to the third connection: we know or estimate the blur kernel and solve a difficult inverse problem under noise.

### How to use the folded solutions / Cómo usar las soluciones plegadas

Each exercise has a **Solution / Solución** cell that is intentionally closed. First try the `TODO`; then open the solution to compare your reasoning with a reference implementation.

> 🇪🇸 La convolución es un operador lineal estructurado que reutiliza el mismo kernel. La convolución verdadera invierte el kernel; la correlación no. Ese mismo operador puede escribirse como una matriz Toeplitz. La convolución transpuesta cambia la forma mediante superposición y suma, pero no deshace automáticamente el operador original. La deconvolución verdadera intenta recuperar una señal perdida y por eso es un problema inverso sensible al ruido.
>
> Cada ejercicio tiene una celda **Solution / Solución** cerrada a propósito. Primero intenta el `TODO`; después abre la solución.

### Learning cycle: Predict → Run → Explain

Before every exercise, predict the output shape and what information should be preserved or lost. Then run it and explain the result.

> 🇪🇸 **Predice → Ejecuta → Explica:** anticipa la forma de salida y qué información debería conservarse o perderse; luego ejecuta y explica el resultado.

## Exercise 1 — convolution, correlation, and Toeplitz on real pixels

We start from two real pieces of the photograph:

- a `160×160` crop for 2D edge filtering;
- a length-32 scanline for the matrix view.

For a `3×3` kernel on a `160×160` image:

- `valid` should produce `158×158`;
- `same` should produce `160×160`;
- `full` should produce `162×162`.

For the scanline, full 1D convolution with a length-3 kernel should produce `32 + 3 - 1 = 34` values.

### What should you try?

1. Compare `correlate2d(patch, sobel_x)` with `convolve2d(patch, flip(sobel_x))`.
2. Verify that they match.
3. Build a Toeplitz matrix `C` for the real scanline and check `C @ scanline == np.convolve(...)`.
4. Move **Mode / Modo** and verify the predicted image sizes.

> 🇪🇸 Usaremos píxeles reales de la fotografía. Comprueba que la correlación con Sobel coincide con la convolución cuando el kernel se invierte, representa una convolución 1D con una matriz Toeplitz y usa el selector de modo para confirmar `valid`, `same` y `full`.

In [ ]:
# TODO
# 1. Compute 2D correlation with sobel_x.
# 2. Compute 2D convolution with np.flip(sobel_x).
# 3. Verify that the two results match.
# 4. Build C = convmtx_full_1d(kernel_1d, len(scanline)).
# 5. Check C @ scanline against np.convolve(scanline, kernel_1d, "full").

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
corr = signal.correlate2d(patch, sobel_x, mode="valid")
conv_flipped = signal.convolve2d(
    patch,
    np.flip(sobel_x),
    mode="valid",
)

print(
    "correlation == convolution with flipped kernel / "
    "correlación == convolución con kernel invertido:",
    np.allclose(corr, conv_flipped),
)

C = convmtx_full_1d(kernel_1d, len(scanline))
via_matrix = C @ scanline
via_convolution = np.convolve(scanline, kernel_1d, mode="full")

print("Toeplitz matrix / matriz Toeplitz:", C.shape)
print("full convolution / convolución full:", via_convolution.shape)
print("C @ scanline == convolution:", np.allclose(via_matrix, via_convolution))

mode_widget = widgets.ToggleButtons(
    options=["valid", "same", "full"],
    value="same",
    description="Mode / Modo:",
    style={"description_width": "95px"},
)

def explore_mode(mode):
    filtered = signal.correlate2d(patch, sobel_x, mode=mode)

    print(
        f"mode/modo={mode} | input/entrada={patch.shape} | "
        f"output/salida={filtered.shape}"
    )

    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
    axes[0].imshow(patch, cmap="gray")
    axes[0].set_title("real patch / recorte real")
    axes[1].imshow(filtered, cmap="gray")
    axes[1].set_title(f"Sobel correlation — {mode}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

mode_output = widgets.interactive_output(
    explore_mode,
    {"mode": mode_widget},
)

display(widgets.VBox([mode_widget, mode_output]))

## Exercise 2 — transposed convolution changes shape, not history

The phrase **“deconvolution layer”** is often used informally for transposed convolution in decoders and generative models. That name is misleading.

Here we use a tiny `2×2` patch sampled from the real photograph so the overlap-add mechanism is visible.

A transposed-convolution-style update places a scaled copy of the kernel into the output for each input value. When stride increases, the output grows and gaps appear between placements.

### What should you try?

1. Take a real `2×2` patch from the photograph.
2. Use a `2×2` all-ones kernel.
3. Implement overlap-add for stride 1.
4. Move **Stride / Paso** from 1 to 3 and inspect the output shape.
5. Explain why a larger output does **not** mean the original pre-convolution image has been recovered.

> 🇪🇸 La convolución transpuesta reutiliza un kernel mediante superposición y suma. Puede aumentar el tamaño espacial, especialmente con stride mayor que 1, pero aumentar la forma no equivale a invertir los valores perdidos por una convolución anterior.

In [ ]:
# TODO
# 1. Extract a real 2×2 patch from img.
# 2. Implement overlap-add with a 2×2 kernel and stride=1.
# 3. Predict the output shape.
# 4. Repeat with stride=2.
# 5. Explain why this is not a true inverse.

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
small = img[250:254:2, 250:254:2].copy()   # real 2×2 pixel patch
ker = np.ones((2, 2), dtype=float)

def transposed_overlap_add(x, kernel, stride=1):
    h, w = x.shape
    kh, kw = kernel.shape

    out_h = (h - 1) * stride + kh
    out_w = (w - 1) * stride + kw
    out = np.zeros((out_h, out_w), dtype=float)

    for i in range(h):
        for j in range(w):
            r = i * stride
            c = j * stride
            out[r:r+kh, c:c+kw] += x[i, j] * kernel

    return out

print("real input / entrada real:")
print(np.round(small, 3))
print("stride=1 output shape / forma:", transposed_overlap_add(small, ker, 1).shape)

stride_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=3,
    step=1,
    description="Stride / Paso:",
    continuous_update=False,
    style={"description_width": "95px"},
)

def explore_transposed(stride):
    out = transposed_overlap_add(small, ker, stride)

    print(
        f"input/entrada={small.shape} | stride/paso={stride} | "
        f"output/salida={out.shape}"
    )
    print(
        "EN: shape expansion is not value inversion; information lost earlier "
        "does not magically return."
    )
    print(
        "ES: aumentar la forma no invierte los valores; la información perdida "
        "antes no reaparece automáticamente."
    )

    fig, axes = plt.subplots(1, 2, figsize=(5.8, 2.8))
    axes[0].imshow(small, cmap="viridis")
    axes[0].set_title("real 2×2 input")
    axes[1].imshow(out, cmap="viridis")
    axes[1].set_title(f"overlap-add, stride={stride}")
    for ax in axes:
        ax.set_xticks(range(ax.images[0].get_array().shape[1]))
        ax.set_yticks(range(ax.images[0].get_array().shape[0]))
    plt.tight_layout()
    plt.show()

transpose_output = widgets.interactive_output(
    explore_transposed,
    {"stride": stride_slider},
)

display(widgets.VBox([stride_slider, transpose_output]))

## Exercise 3 — true deconvolution on a real photograph

Now we solve an actual inverse problem.

We blur a real `256×256` crop with a `9×9` point-spread function (PSF), add a small controlled amount of noise, and attempt to recover the original with **Richardson–Lucy deconvolution**.

There is an important measurement trap: boundary pixels are where the algorithm has the least information about what lies outside the image. We therefore compute the error after removing a fixed border.

### What should you try?

1. Blur the real image crop with a normalized `9×9` averaging PSF.
2. Add small reproducible Gaussian noise.
3. Recover it with `richardson_lucy`.
4. Measure relative error on the interior only.
5. Move **Iterations / Iteraciones** and observe the trade-off: too few iterations under-correct; many iterations can begin to emphasize noise.

> 🇪🇸 Ahora sí resolvemos un problema de deconvolución. Desenfocamos un recorte real con una PSF conocida, añadimos ruido controlado y usamos Richardson–Lucy para recuperar detalle. Medimos únicamente el interior porque los bordes contienen artefactos propios de la falta de información fuera de la imagen.

In [ ]:
# TODO
# 1. Build a normalized 9×9 averaging PSF.
# 2. Blur work with signal.fftconvolve(..., mode="same").
# 3. Add small reproducible noise.
# 4. Recover with richardson_lucy.
# 5. Compare interior relative error before and after.

In [ ]:
#@title Solution / Solución — open after trying / abre después de intentar { display-mode: 'form' }
psf = np.ones((9, 9), dtype=float)
psf /= psf.sum()

blurred = signal.fftconvolve(work, psf, mode="same")
noise = 0.002 * np.random.default_rng(0).standard_normal(work.shape)
noisy = np.clip(blurred + noise, 0, 1)

border = 20

def interior_relative_error(candidate):
    ref = work[border:-border, border:-border]
    cand = candidate[border:-border, border:-border]
    return np.linalg.norm(cand - ref) / np.linalg.norm(ref)

recovered_20 = richardson_lucy(noisy, psf, num_iter=20, clip=False)

print(
    "blurred+noise error / error desenfoque+ruido:",
    f"{interior_relative_error(noisy):.4f}",
)
print(
    "RL 20 iterations / iteraciones:",
    f"{interior_relative_error(recovered_20):.4f}",
)

iterations_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=50,
    step=3,
    description="Iterations / Iteraciones:",
    continuous_update=False,
    style={"description_width": "145px"},
)

def explore_deconvolution(iterations):
    recovered = richardson_lucy(
        noisy,
        psf,
        num_iter=iterations,
        clip=False,
    )

    err_before = interior_relative_error(noisy)
    err_after = interior_relative_error(recovered)

    print(
        f"iterations/iteraciones={iterations} | "
        f"before/antes={err_before:.4f} | after/después={err_after:.4f}"
    )

    if err_after < err_before:
        print("EN: recovery improved the interior error.")
        print("ES: la recuperación redujo el error interior.")
    else:
        print("EN: at this iteration count the recovery no longer improves the metric.")
        print("ES: con este número de iteraciones la recuperación ya no mejora la métrica.")

    fig, axes = plt.subplots(1, 3, figsize=(8.6, 3.0))
    images = [work, noisy, np.clip(recovered, 0, 1)]
    titles = [
        "original real / original",
        "blurred + noise / desenfoque",
        f"Richardson–Lucy ({iterations})",
    ]

    for ax, im, title in zip(axes, images, titles):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

deconv_output = widgets.interactive_output(
    explore_deconvolution,
    {"iterations": iterations_slider},
)

display(widgets.VBox([iterations_slider, deconv_output]))

## What just happened

You followed one operator from forward use to inverse use.

1. **Correlation vs convolution:** deep-learning libraries usually slide the learned kernel without flipping it, which is cross-correlation. True convolution gives the same result when the kernel is flipped first.
2. **Toeplitz view:** a convolution of real pixel measurements became an ordinary matrix product `C @ x`. The special part was the repeated structure of `C`, not a new algebra.
3. **Transposed convolution:** overlap-add changed the spatial shape using the same local weights. It behaved like the transpose/adjoint partner of a convolutional operator, not like a true inverse.
4. **True deconvolution:** Richardson–Lucy used the known blur kernel plus an iterative model to recover some detail from a noisy real image. The result had to be measured away from unreliable boundaries.

### The sentence to remember

> **Convolution applies a structured operator; transposed convolution applies its shape-changing partner; deconvolution tries to solve the inverse problem.**

In microscopy, astronomy, and medical imaging, the blur kernel is often described by a **point-spread function (PSF)**. Deconvolution is useful precisely because imaging systems spread information before we ever see the pixels.

> 🇪🇸 Seguiste un mismo operador desde el problema directo hasta el inverso. La correlación y la convolución se diferencian por el volteo del kernel; Toeplitz muestra que la operación sigue siendo multiplicación matricial; la convolución transpuesta cambia la forma pero no recupera automáticamente lo perdido; y Richardson–Lucy intenta resolver un problema inverso real bajo ruido.
>
> **Frase para recordar:** la convolución aplica un operador estructurado; la convolución transpuesta aplica su compañero que cambia la forma; la deconvolución intenta resolver el problema inverso.

---

## Done with this section

Next up: **10 · Tucker decomposition on real data** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)